In [12]:
# Basic Libraries / Import statements

import numpy as np

import plotly.graph_objects as go 

In [13]:
# Target: assumed to be cruising in straight line at high altitude
target_state = np.array([0.0, 0.0 , 10000.0, 300.0, 0.0, 0.0])

# For basic assume air to ground at a upward angle of attack 
interceptor_state = np.array([2000.0, 500.0, 0.0, -150.0, -50.0, 600.0])

# Basic physics 
dt = 0.1  

# Sim for 30 seconds just to verify 
total_time = 30  
steps = int(total_time / dt)

target_path = np.zeros((steps, 3))
interceptor_path = np.zeros((steps, 3))

# Simple Euler integration (constant velocity for this initial test)

for i in range(steps):
    # Recording positions
    target_path[i] = target_state[0:3]
    interceptor_path[i] = interceptor_state[0:3]
    
    # Update positions based on velocity
    target_state[0:3] += target_state[3:6] * dt
    interceptor_state[0:3] += interceptor_state[3:6] * dt

# Visualize
fig = go.Figure()

# Plot Target Trajectory
fig.add_trace(go.Scatter3d(
    x=target_path[:,0], y=target_path[:,1], z=target_path[:,2],
    mode='lines+markers',
    marker=dict(size=2, color='red'),
    line=dict(color='red', width=2),
    name='Target'
))

# Plot Interceptor Trajectory
fig.add_trace(go.Scatter3d(
    x=interceptor_path[:,0], y=interceptor_path[:,1], z=interceptor_path[:,2],
    mode='lines+markers',
    marker=dict(size=2, color='blue'),
    line=dict(color='blue', width=2),
    name='Interceptor'
))

fig.update_layout(
    title='Phase 1: 3D Kinematic Environment',
    scene=dict(
        xaxis_title='X Position (m)',
        yaxis_title='Y Position (m)',
        zaxis_title='Altitude Z (m)'
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()


In [14]:

# Intial states, forcing collision 
target_state = np.array([0.0, 0.0, 10000.0, 300.0, 0.0, 0.0]) 

# Interceptor velocity 
# adjusted to ensure an interception at t=20s

interceptor_state = np.array([2000.0, 500.0, 0.0, 200.0, -25.0, 500.0])

dt = 0.1  
total_time = 30  
steps = int(total_time / dt)

target_path = np.zeros((steps, 3))
interceptor_path = np.zeros((steps, 3))

# Sim and Distance Tracking 
distances = np.zeros(steps)

for i in range(steps):
    target_path[i] = target_state[0:3]
    interceptor_path[i] = interceptor_state[0:3]
    
    # Calculate distance between the two missiles at this exact time step
    distances[i] = np.linalg.norm(target_path[i] - interceptor_path[i])
    
    target_state[0:3] += target_state[3:6] * dt
    interceptor_state[0:3] += interceptor_state[3:6] * dt

# Interception point
hit_radius = 50.0 # meters
min_dist_idx = np.argmin(distances)
closest_distance = distances[min_dist_idx]
time_of_interception = min_dist_idx * dt
interception_coords = target_path[min_dist_idx]

print(f"--- Simulation Results ---")
print(f"Closest Approach: {closest_distance:.2f} meters")
if closest_distance <= hit_radius:
    print(f"STATUS: SUCCESSFUL INTERCEPTION at t = {time_of_interception:.1f}s")
    print(f"Coordinates (x, y, z): {interception_coords}")
else:
    print(f"STATUS: MISS")


fig = go.Figure()

# Plot static tracks (faded lines so you can see the path before the dot gets there)
fig.add_trace(go.Scatter3d(x=target_path[:,0], y=target_path[:,1], z=target_path[:,2], mode='lines', line=dict(color='rgba(255,0,0,0.2)', width=2), name='Target Track'))
fig.add_trace(go.Scatter3d(x=interceptor_path[:,0], y=interceptor_path[:,1], z=interceptor_path[:,2], mode='lines', line=dict(color='rgba(0,0,255,0.2)', width=2), name='Interceptor Track'))

# Plot the specific interception point as a large marker
if closest_distance <= hit_radius:
    fig.add_trace(go.Scatter3d(
        x=[interception_coords[0]], y=[interception_coords[1]], z=[interception_coords[2]],
        mode='markers', marker=dict(size=8, color='yellow', symbol='diamond'), name='Point of Impact'
    ))

# Create the animated dots (starting at t=0)
fig.add_trace(go.Scatter3d(x=[target_path[0,0]], y=[target_path[0,1]], z=[target_path[0,2]], mode='markers', marker=dict(size=6, color='red'), name='Target'))
fig.add_trace(go.Scatter3d(x=[interceptor_path[0,0]], y=[interceptor_path[0,1]], z=[interceptor_path[0,2]], mode='markers', marker=dict(size=6, color='blue'), name='Interceptor'))

# Build frames for the animation (skipping frames so the browser doesn't freeze)
frame_skip = 5 
frames = []
for k in range(0, steps, frame_skip):
    frames.append(go.Frame(
        data=[
            go.Scatter3d(x=[target_path[k,0]], y=[target_path[k,1]], z=[target_path[k,2]]),
            go.Scatter3d(x=[interceptor_path[k,0]], y=[interceptor_path[k,1]], z=[interceptor_path[k,2]])
        ],
        traces=[3, 4], # Update only the animated dots (traces 3 and 4)
        name=f'frame_{k}'
    ))
fig.frames = frames

# Add Play/Pause buttons
fig.update_layout(
    title='Phase 1: Animated Kinematics & Interception',
    scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Altitude (m)'),
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=50, redraw=True), fromcurrent=True)]),
            dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])
        ]
    )]
)

fig.show()

--- Simulation Results ---
Closest Approach: 0.00 meters
STATUS: SUCCESSFUL INTERCEPTION at t = 20.0s
Coordinates (x, y, z): [ 6000.     0. 10000.]


# **Phase one working so far**
### Still have to do these updates pre phase 2
#### *Updates:*

- Physics behind actual trajectory and position -> need to understand / do 
- Vector Positioning in relation to missile dynamics 
- Collision Geometry 
- Point of Impact physics, i.e. one collision what happens 
- Open sourced missile information 
- Actual interception math  -> test against forced values 
- Actual missile atypcial flight patterns and normal/varied impact patterns 
- Basic enviorment mapping 
- Atypical Launch :
    Patterns 
    Timings for intercept to target 
    Cruise patterns 

## Updates Planned 

- using a self made Rk4 
- Rk4 -> is a Runge Kutta (4th order)
- Numerical Method used for solving differential equations 

- The physics gives a rate of change
- but the sim actually need positional information 
- Since it is very difficult to solve for it makes more senst to integrate numericaly 
- 

In [15]:

def rk4_step(state, t, dt, derivatives_fn):
    """
    Standard 4th-Order Runge-Kutta integrator.
    Updates the state vector for a single time step.
    """
    k1 = derivatives_fn(state, t)
    k2 = derivatives_fn(state + 0.5 * dt * k1, t + 0.5 * dt)
    k3 = derivatives_fn(state + 0.5 * dt * k2, t + 0.5 * dt)
    k4 = derivatives_fn(state + dt * k3, t + dt)
    
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)

def missile_dynamics(state, t):
    """
    Calculates the derivatives of the state vector.
    State:      [x, y, z, vx, vy, vz]
    Derivative: [vx, vy, vz, ax, ay, az]
    """
    vel = state[3:6]
    
    # 1. Gravity Force (assuming flat earth for Phase 1)
    # Accelerates the z-axis downwards at 9.81 m/s^2
    gravity_accel = np.array([0.0, 0.0, -9.81])
    
    # 2. Aerodynamic Drag (Placeholder for Sprint 1)
    # Drag opposes the velocity vector. We will use a simple linear coefficient
    # for now before bringing in air density and Mach cross-sections.
    drag_coeff = 0.005 
    drag_accel = -drag_coeff * vel * np.linalg.norm(vel) # Proportional to v^2
    
    # 3. Total Acceleration
    accel = gravity_accel + drag_accel
    
    # The derivative of position is velocity, the derivative of velocity is acceleration
    return np.concatenate((vel, accel))

In [16]:
target_state = np.array([0.0, 0.0, 10000.0, 300.0, 0.0, 0.0]) 
interceptor_state = np.array([2000.0, 500.0, 0.0, 200.0, -25.0, 560.0])

dt = 0.1  
total_time = 30  
steps = int(total_time / dt)

target_path = np.zeros((steps, 3))
interceptor_path = np.zeros((steps, 3))
distances = np.zeros(steps)

# --- 3. Simulation Loop (RK4 Integration) ---
for i in range(steps):
    target_path[i] = target_state[0:3]
    interceptor_path[i] = interceptor_state[0:3]
    
    distances[i] = np.linalg.norm(target_path[i] - interceptor_path[i])
    
    t = i * dt
    
    target_state = rk4_step(target_state, t, dt, missile_dynamics)
    interceptor_state = rk4_step(interceptor_state, t, dt, missile_dynamics)

# --- 4. Interception Analysis ---
hit_radius = 50.0 
min_dist_idx = np.argmin(distances)
closest_distance = distances[min_dist_idx]
time_of_interception = min_dist_idx * dt
interception_coords = target_path[min_dist_idx]

print(f"--- Simulation Results ---")
print(f"Closest Approach: {closest_distance:.2f} meters")
if closest_distance <= hit_radius:
    print(f"STATUS: SUCCESSFUL INTERCEPTION at t = {time_of_interception:.1f}s")
    print(f"Coordinates (x, y, z): {interception_coords}")
else:
    print(f"STATUS: MISS")

# --- 5. 3D Animated Visualization ---
fig = go.Figure()

fig.add_trace(go.Scatter3d(x=target_path[:,0], y=target_path[:,1], z=target_path[:,2], mode='lines', line=dict(color='rgba(255,0,0,0.2)', width=2), name='Target Track'))
fig.add_trace(go.Scatter3d(x=interceptor_path[:,0], y=interceptor_path[:,1], z=interceptor_path[:,2], mode='lines', line=dict(color='rgba(0,0,255,0.2)', width=2), name='Interceptor Track'))

if closest_distance <= hit_radius:
    fig.add_trace(go.Scatter3d(
        x=[interception_coords[0]], y=[interception_coords[1]], z=[interception_coords[2]],
        mode='markers', marker=dict(size=8, color='yellow', symbol='diamond'), name='Point of Impact'
    ))

fig.add_trace(go.Scatter3d(x=[target_path[0,0]], y=[target_path[0,1]], z=[target_path[0,2]], mode='markers', marker=dict(size=6, color='red'), name='Target'))
fig.add_trace(go.Scatter3d(x=[interceptor_path[0,0]], y=[interceptor_path[0,1]], z=[interceptor_path[0,2]], mode='markers', marker=dict(size=6, color='blue'), name='Interceptor'))

frame_skip = 5 
frames = []
for k in range(0, steps, frame_skip):
    frames.append(go.Frame(
        data=[
            go.Scatter3d(x=[target_path[k,0]], y=[target_path[k,1]], z=[target_path[k,2]]),
            go.Scatter3d(x=[interceptor_path[k,0]], y=[interceptor_path[k,1]], z=[interceptor_path[k,2]])
        ],
        traces=[3, 4], 
        name=f'frame_{k}'
    ))
fig.frames = frames

fig.update_layout(
    title='Phase 1: Animated Kinematics (RK4 Dynamics)',
    scene=dict(xaxis_title='X (m)', yaxis_title='Y (m)', zaxis_title='Altitude (m)'),
    updatemenus=[dict(
        type="buttons",
        buttons=[
            dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=50, redraw=True), fromcurrent=True)]),
            dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")])
        ]
    )]
)

fig.show()

--- Simulation Results ---
Closest Approach: 9472.71 meters
STATUS: MISS
